# Preserve existing assignments in an A/A split

[All How-to Guides](README.md)

Some users are already in control or treatment. Keep their assignments and allocate the remaining users in the same run.

## 1. Mark fixed assignments

In the column with `ConstGroupRole`, enter `control` for control and `test` or `test_1` for the first treatment group. For multiple variants, use `test_2` and subsequent labels from the configured split. Leave unassigned rows as `None`.

In [1]:
import pandas as pd

from hypex import AATest
from hypex.dataset import ConstGroupRole, Dataset, TargetRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=240, dependent_division=False, random_state=42
)[["Target_lag_1"]]
frame = frame.assign(pinned_group=None)
frame.loc[:19, "pinned_group"] = "control"
frame.loc[20:39, "pinned_group"] = "test"
data = Dataset(
    roles={"Target_lag_1": TargetRole(), "pinned_group": ConstGroupRole(str)},
    data=frame,
)
frame["pinned_group"].fillna("free").value_counts()

pinned_group
free       200
control     20
test        20
Name: count, dtype: int64

Each group has 20 fixed assignments, and 200 users are unassigned. The label `free` appears only in the display.

## 2. Run the split and check the assignments

`control_size` sets the target control share for the whole table. HypEx accounts for fixed assignments when allocating the remaining users. Compare the original labels with the `split` column.

`ConstGroupRole` identifies fixed assignments. The example spells out the default quota with `control_size=0.5` and fixes a single candidate split with `random_states=[42]`. These parameters govern how unassigned users are allocated.

Select the two columns with `Dataset.select()`. The `data` property gives you the pandas DataFrame for `crosstab`.

In [2]:
result = AATest(control_size=0.5, random_states=[42]).execute(data)
assignments = result.best_split.select(["pinned_group", "split"]).data
pd.crosstab(assignments["pinned_group"].fillna("free"), assignments["split"])

split,control,test_1
pinned_group,,
control,20,0
free,95,105
test,0,20


All fixed rows kept their assignments, and the remaining users were allocated across the two groups. If fixed assignments already exceed a group's quota, keeping those assignments makes the target share unattainable.

The [Spark tutorial](../quick_tutorials/SparkBackend.ipynb#spark-fixed-groups) runs this example on both backends.